# Restaurants - classification problem

### Imports and setup

In [21]:
# Cell 1: Imports and Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression as LRSelector
from sklearn.preprocessing import TargetEncoder
from sklearn.base import clone

# Imblearn for SMOTE and Undersampling inside the pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

### Load Data

In [22]:
# Cell 2: Data Loading and Advanced Feature Engineering

# 1. Load data
train_df = pd.read_csv('data/restaurants_train.csv')
test_df = pd.read_csv('data/restaurants_test.csv')

# 2. Feature Engineering Function
def engineer_and_drop(df):
    df_eng = df.copy()
    
    # Has Recent Activity flag
    df_eng['has_recent_activity'] = (~df_eng['ratings_num_1m_prior'].isna() & (df_eng['ratings_num_1m_prior'] > 0)).astype(int)
    
    # Temporal Information 
    df_eng['rating_momentum'] = df_eng['ratings_avg_1m_prior'] - df_eng['ratings_avg_12m_prior']
    df_eng['rating_vol_trend'] = df_eng['ratings_num_1m_prior'] - df_eng['ratings_num_3m_prior']
    
    # Spatial Scale Information
    df_eng['competition_density_ratio'] = df_eng['catch_restaurant_count_500m'] / (df_eng['catch_restaurant_count_2000m'] + 1)
    df_eng['local_quality_advantage'] = df_eng['rating_avg'] - df_eng['catch_rating_avg_500m']
    df_eng['age_vs_competition'] = df_eng['place_age_days'] - df_eng['catch_place_age_days_500m']
    
    # Rating Distribution Shape
    df_eng['rating_polarization'] = (df_eng['rating_1'] + df_eng['rating_5']) / (df_eng['user_ratings_total'] + 1)
    df_eng['rating_negative_share'] = (df_eng['rating_1'] + df_eng['rating_2']) / (df_eng['user_ratings_total'] + 1)
    
    # Review Text Engagement
    df_eng['engaged_review_count'] = df_eng['user_ratings_total'] * df_eng['review_has_text_pct']
    
    # Tourist vs. Local Gap
    df_eng['local_tourist_rating_gap'] = df_eng['rating_pl'].fillna(df_eng['rating_avg']) - \
                                         df_eng['rating_foreign'].fillna(df_eng['rating_avg'])
    
    # POI Gradient
    df_eng['poi_cluster_ratio'] = df_eng['poi_count_500m'] / (df_eng['poi_count_2000m'] + 1)
    
    # Interaction
    df_eng['income_unemp_interaction'] = df_eng['gus_64428'] * df_eng['gus_79214']

    df_eng['category_top20'] = df_eng['category_top20'].astype(str).replace('nan', 'Unknown')
    
    # Only drop features directly consumed by engineered features.
    # Keep intermediate spatial scales (1000m) and temporal features;
    # let L1 regularization handle remaining redundancy.
    cols_to_drop = [
        'ratings_num_3m_prior', 'ratings_num_12m_prior',
        'ratings_avg_12m_prior',
        'catch_restaurant_count_2000m', 'catch_rating_avg_2000m', 'catch_place_age_days_2000m',
        'rating_5', 'rating_1', 'rating_2',
        'rating_pl', 'rating_foreign',
        'poi_count_2000m'
    ]
    return df_eng.drop(columns=cols_to_drop, errors='ignore')

# 3. Apply transformations
train_df_eng = engineer_and_drop(train_df)
test_df_eng = engineer_and_drop(test_df)
print(f"Engineered train shape: {train_df_eng.shape}")

### Preprocessing

In [23]:
# Cell 3: Setup Preprocessors and Split

# 1. Separate features and target
X_raw = train_df_eng.drop(columns=['restaurant_id', 'status_closed'])
y_raw = train_df_eng['status_closed']
X_test_kaggle = test_df_eng.drop(columns=['restaurant_id'])

# 1. Group the columns
target_enc_col = ['category_top20'] 
cat_cols = ['price_level', 'has_photo', 'type_meal_takeaway', 
            'type_meal_delivery', 'type_bar', 'type_cafe', 'type_night_club', 
            'weekends_only', 'workdays_only', 'affiliated']
num_cols = [col for col in X_raw.columns if col not in cat_cols and col not in target_enc_col]

for col in cat_cols + target_enc_col:
    X_raw[col] = X_raw[col].fillna('Unknown').astype(str)
    X_test_kaggle[col] = X_test_kaggle[col].fillna('Unknown').astype(str)

# 2. Build the component pipelines
numeric_transformer = ImbPipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)), 
    ('power_transform', PowerTransformer(method='yeo-johnson')) 
])

categorical_transformer = ImbPipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

target_encoding_transformer = ImbPipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('target_enc', TargetEncoder(target_type='binary'))
])

# 3. Combine them into the Master Preprocessor
master_preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols),
    ('target', target_encoding_transformer, target_enc_col)
])

# KNN Specific Preprocessor (MinMaxScaler instead of PowerTransformer)
numeric_transformer_knn = ImbPipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)), 
    ('scaler', MinMaxScaler()) 
])
preprocessor_knn = ColumnTransformer(transformers=[
    ('num', numeric_transformer_knn, num_cols),
    ('cat', categorical_transformer, cat_cols),
    ('target', target_encoding_transformer, target_enc_col) 
])

### Train/Test split

In [24]:
# split the data randomly into train (80%) and validation (20%)
# we use stratify to maintain the 10:1 target distribution
X_train, X_val, y_train, y_val = train_test_split(
    X_raw, y_raw, test_size=0.2, stratify=y_raw, random_state=123
)

# Cross-validation strategy
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
print("Data successfully preprocessed and split!")

print(f"Training set size: {X_train.shape}")
print(f"Validation set size: {X_val.shape}")

Data successfully preprocessed and split!
Training set size: (26636, 62)
Validation set size: (6660, 62)


## Defining and Training the Models

### Logistic Regression / Elastic net

In [ ]:
# Cell 4: Elastic Net Model

elastic_pipeline = ImbPipeline(steps=[
    ('preprocessor', master_preprocessor),
    ('smote', SMOTE(random_state=123)),
    ('model', LogisticRegression(solver='saga', max_iter=10000, random_state=123))
])

grid_elastic = GridSearchCV(
    estimator=elastic_pipeline,
    param_grid={'model__C': [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0], 'model__l1_ratio': [0, 0.1, 0.5, 0.7, 0.9, 1.0]},
    scoring='balanced_accuracy',
    cv=cv5,
    n_jobs=-1
)

print("Training Elastic Net...")
grid_elastic.fit(X_train, y_train)
print(f"Best Hyperparameters: {grid_elastic.best_params_}")
print(f"Elastic Net CV Balanced Accuracy: {grid_elastic.best_score_:.4f}")

### K-nearest neighbours (KNN)

In [ ]:
# Cell 5: KNN Model

knn_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor_knn),
    ('selector', SelectFromModel(LRSelector(l1_ratio=1.0, solver='saga', C=0.1, max_iter=5000, random_state=123))),
    ('smote', SMOTE(random_state=123)),
    ('model', KNeighborsClassifier())
])

grid_knn = GridSearchCV(
    estimator=knn_pipeline,
    param_grid={
        'model__n_neighbors': [15, 30, 50],
        'model__weights': ['uniform', 'distance'],
        'model__p': [1, 2] 
    },
    scoring='balanced_accuracy',
    cv=cv5,
    n_jobs=-1
)

print("Training KNN (with L1 feature selection)...")
grid_knn.fit(X_train, y_train)
print(f"Best Hyperparameters: {grid_knn.best_params_}")
print(f"KNN CV Balanced Accuracy: {grid_knn.best_score_:.4f}")

### Support Vector Machine (SVM)

In [27]:
# Cell 6: SVM Model

svm_pipeline = ImbPipeline(steps=[
    ('preprocessor', master_preprocessor),
    #('undersampler', RandomUnderSampler(random_state=123)), # Mandatory for SVM speed, remove class_weight='balanced' if uncommented
    ('model', SVC(random_state=123, probability=True, class_weight='balanced'))])

grid_svm = GridSearchCV(
    estimator=svm_pipeline,
    param_grid={
        'model__C': [0.1, 1.0, 10.0],
        'model__gamma': ['scale', 'auto'],
        'model__kernel': ['rbf'] 
    },
    scoring='balanced_accuracy',
    cv=cv5,
    n_jobs=-1
)

print("Training SVM...")
grid_svm.fit(X_train, y_train)
print(f"Best Hyperparameters: {grid_svm.best_params_}")
print(f"SVM CV Balanced Accuracy: {grid_svm.best_score_:.4f}")

Training SVM...
Best Hyperparameters: {'model__C': 0.1, 'model__gamma': 'auto', 'model__kernel': 'rbf'}
SVM CV Balanced Accuracy: 0.6667


### SVM with RandomUnderSampler

In [ ]:
# Undersampled SVM: faster training and often better balanced accuracy
svm_under_pipeline = ImbPipeline(steps=[
    ('preprocessor', clone(master_preprocessor)),
    ('undersampler', RandomUnderSampler(random_state=123)),
    ('model', SVC(random_state=123, probability=True))
])

grid_svm_u = GridSearchCV(
    estimator=svm_under_pipeline,
    param_grid={
        'model__C': [0.1, 1.0, 10.0],
        'model__gamma': ['scale', 'auto'],
        'model__kernel': ['rbf']
    },
    scoring='balanced_accuracy',
    cv=cv5,
    n_jobs=-1
)

print("Training Undersampled SVM...")
grid_svm_u.fit(X_train, y_train)
print(f"Best Hyperparameters: {grid_svm_u.best_params_}")
print(f"SVM (Undersampled) CV Balanced Accuracy: {grid_svm_u.best_score_:.4f}")

### Model Evaluaton

In [28]:
# Cell 7: Leak-Free Threshold Tuning & Final Kaggle Export

searches = {
    'Elastic Net': grid_elastic,
    'KNN': grid_knn,
    'SVM (RBF, balanced)': grid_svm,
    'SVM (RBF, undersampled)': grid_svm_u
}

best_overall_name = ""
best_overall_score = 0
best_model = None

# Automatically crown the winner
for name, search_obj in searches.items():
    if search_obj.best_score_ > best_overall_score:
        best_overall_score = search_obj.best_score_
        best_model = search_obj.best_estimator_
        best_overall_name = name

print("-" * 50)
print(f"BEST MODEL: {best_overall_name}")
print(f"Best CV Balanced Accuracy: {best_overall_score:.4f}")
print("-" * 50)

# FIX 3: Leak-Free Threshold Tuning!
# We use cross_val_predict on X_train to get unbiased, out-of-fold probabilities.
print("\nCalculating Out-Of-Fold probabilities...")
oof_train_probs = cross_val_predict(best_model, X_train, y_train, cv=5, method='predict_proba', n_jobs=-1)[:, 1]

thresholds = np.linspace(0.2, 0.8, 100)
best_threshold = 0.5
best_train_bacc = 0

# Optimize threshold strictly on the out-of-fold training data
for t in thresholds:
    custom_preds = (oof_train_probs >= t).astype(int)
    score = balanced_accuracy_score(y_train, custom_preds)
    if score > best_train_bacc:
        best_train_bacc = score
        best_threshold = t

print(f"Optimal Threshold (Found via CV on Train): {best_threshold:.3f}")

# Now we securely evaluate on the untouched Validation Set
val_probs = best_model.predict_proba(X_val)[:, 1]
final_val_preds = (val_probs >= best_threshold).astype(int)

print(f"Validation Balanced Accuracy (Default 0.5 threshold): {balanced_accuracy_score(y_val, best_model.predict(X_val)):.4f}")
print(f"Validation Balanced Accuracy (OPTIMIZED Threshold): {balanced_accuracy_score(y_val, final_val_preds):.4f}")

# ---------------------------------------------------------
# FINAL KAGGLE SUBMISSION PREPARATION
# ---------------------------------------------------------
print(f"\nRetraining {best_overall_name} on the FULL training dataset...")
best_model.fit(X_raw, y_raw)

# Generate probabilities for the Kaggle test set
test_probs = best_model.predict_proba(X_test_kaggle)[:, 1]

# Apply our highly optimized threshold
final_test_predictions = (test_probs >= best_threshold).astype(int)

# Create submission dataframe
submission = pd.DataFrame({
    'restaurant_id': test_df['restaurant_id'],
    'status_closed': final_test_predictions
})

# Save to CSV
submission_filename = f'submission_best_{best_overall_name.replace(" ", "_")}.csv'
submission.to_csv(submission_filename, index=False)
print(f"Submission saved to {submission_filename}!")